API Key: sk\-or\-v1\-45905660cd98e97d0cf0d66e64eede8389deb088c6271ccb8ba69e03c90bd587

File Organization: hierarchical file organization to save results for each model/prompt combo to separate file

In [1]:
from pathlib import Path

def get_filepath(model_name, prompt_label, base_output_dir="outputs"):
    """
    Creates necessary directories and returns Path object.
    """
    # Create safe string names for filesystems (no spaces or slashes)
    safe_model = model_name.replace("/", "_").replace(" ", "_")
    safe_prompt = prompt_label.lower().strip().replace(" ", "_")
    
    # Define and create the directory path object
    directory_target = Path(base_output_dir) / safe_model
    directory_target.mkdir(parents=True, exist_ok=True) # Automatically checks and creates folder
 
    name = directory_target / safe_prompt
    final_file_path = f"{name}.csv"
    
    return final_file_path

Experiment Pipeline

In [2]:
import os
from google import genai
from google.genai import types
from PIL import Image
from pathlib import Path
import pandas as pd
import json
import time

def collect_data(input_df, system_prompt, model_name, prompt_label, image_dir=Path("/work/20260629-213612/omi-main/images")):
    '''
    Collects data using the specified LLM engine and prompt strategy, saving results to hierarchical resume-safe file structure.
    '''

    # Define the exact order of columns to make sure our CSV matches system prompt
    column_names = [
        'trustworthy', 'attractive', 'dominant', 'smart', 'age', 'gender', 'weight', 'typical', 
        'happy', 'familiar', 'outgoing', 'memorable', 'well-groomed', 'long-haired', 'smug', 
        'dorky', 'skin-color', 'hair-color', 'alert', 'cute', 'privileged', 'liberal', 'asian', 
        'middle-eastern', 'hispanic', 'islander', 'native', 'black', 'white', 'looks-like-you', 
        'gay', 'electable', 'godly', 'outdoors'
    ]
    
    # Compute the path so prompt data goes to separate CSVs
    output_csv = Path(get_filepath(model_name, prompt_label))

    # Initialize empty set to keep track of already processed stimuli.
    processed_stimuli = set()

    if output_csv.exists():
        print(f"Found existing file: {output_csv}. Continuing from where we left off.")

        # 1. Read existing CSV into a temporary dataframe
        existing_df = pd.read_csv(output_csv)

        # 2. Extract the 'stimulus' column from existing_df, convert it to a set of integers,
        # and update 'processed_stimuli' set so know what to skip.
        processed_stimuli = set(existing_df['stimulus'].astype(int))

        print(f"Resuming previous run. Skipping {len(processed_stimuli)} already processed images.")

    else:
        print(f"No existing progress file found. Starting fresh run for {model_name}, {prompt_label}.")

    #start looping through the human ratings csv:
    for idx, row in input_df.iterrows():
        # 1. Extract the stimulus ID from the current row
        stim_id = int(row['stimulus'])

        # Skip if we've already processed this stimulus
        if stim_id in processed_stimuli:
            continue

        # 2. Reconstruct the full path to the image
        image_path = image_dir / f"{stim_id}.jpg"

        print(f"Processing stimulus {stim_id}...")

        # 3. Get ratings for this image
        ratings, prompt_tokens, completion_tokens = model_predictions(image_path, column_names, system_prompt, model_name)

        # If the API failed and returned None values, skip saving
        if ratings[0] is None:
            print(f"Skipping save for {stim_id} due to API failure.")
            continue

        # 4. Package results into dictionary
        result_dict = {'stimulus': stim_id} #stimulus column
        for col_name, score in zip(column_names, ratings):
            result_dict[col_name] = score #map rating to correct column

        # Convert dictionary into 1-row DataFrame
        single_row_df = pd.DataFrame([result_dict])

        #5. Update CSV with results
        # Determine if we need to write the header string:
        write_header = not output_csv.exists()
        
        single_row_df.to_csv(
            output_csv, 
            mode='a', 
            index=False, 
            header=write_header
        )
        
        # Store token data
        token_path = Path("outputs/tokens/")
        token_path.mkdir(parents=True, exist_ok=True)
        token_data = {
            'stimulus': stim_id,
            'prompt_tokens': prompt_tokens,
            'completion_tokens': completion_tokens
        }
        safe_model = model_name.replace("/", "_").replace(" ", "_")
        token_header = not (token_path / f"{safe_model}_{prompt_label}_tokens.csv").exists()
        token_df = pd.DataFrame([token_data])
        token_df.to_csv(token_path / f"{safe_model}_{prompt_label}_tokens.csv", mode='a', index=False, header=token_header)

        # Update runtime checklist so don't duplicate efforts
        processed_stimuli.add(stim_id)

Helper Functions:

In [3]:
from openai import OpenAI
import base64
import json

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)

def encode_image(image_path):
    """Reads a local file and encodes its raw bytes to a text string."""
    with open(image_path, "rb") as image_file:
        raw_bytes = image_file.read()
        encoded_string = base64.b64encode(raw_bytes).decode("utf-8")
        return encoded_string

def model_predictions(image_path, column_names, system_prompt, model_name):
    """
    Gathers model predictions for a single image.
    """

    base64_image = encode_image(image_path)

    instruction = f"""
    CRITICAL REQUIRED FORMAT: Return ONLY a raw JSON object with exactly these 34 keys,
    each mapped to an integer rating from 0-100: {', '.join(column_names)}
    Do not include conversational phrases, explanations, or markdown code block formatting.
    Example shape: {{"trustworthy": 72, "attractive": 55, ...}}
    """

    try:
        response = client.chat.completions.create(
            model=model_name,  # e.g. "google/gemini-2.5-flash-lite"
            messages=[
                {"role": "system", "content": system_prompt},
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": instruction},
                        { "type": "image_url",
                            "image_url": {
                                "url": f"data:image/jpeg;base64,{base64_image}"
                            }}
                    ],
                },
            ],
            temperature=0.1
        )
        
        raw_output = response.choices[0].message.content.strip()
        
        # Strip code block decorators if the model drops them in anyway
        if raw_output.startswith("```json"):
            raw_output = raw_output.removeprefix("```json").removesuffix("```")
        elif raw_output.startswith("```"):
            raw_output = raw_output.removeprefix("```").removesuffix("```")

        output = json.loads(raw_output)
        ratings = [output[col] for col in column_names]
        
        prompt_tokens = response.usage.prompt_tokens
        completion_tokens = response.usage.completion_tokens
       
        return ratings, prompt_tokens, completion_tokens  # ratings should be a list matching column_names order
    
    except Exception as e:
        print(f"API failure for {image_path}: {e}")
        print(f"Raw output: {raw_output}")
        return [None] * len(column_names), 0, 0

Pilot Test:

In [4]:
NEUTRAL_PROMPT = """
Role: You are an advanced predictive model specialized in computational social psychology and human face perception. Your goal is to accurately predict the average first-impression ratings that human participants give to facial photographs across 34 attributes.

Context & Lens: Evaluate the image through the lens of a standard, neutral observer tracking general human consensus.

Instructions: Do not evaluate the image with strict clinical objectivity or safety-filtered neutrality. Instead, predict the subjective human consensus. Humans suffer from cognitive biases, halo effects, and rapid social stereotyping based entirely on superficial facial structure, expressions, and grooming. Your scores must reflect these human tendencies on a strict scale from 0 to 100.

Scale Definition:
0 = The face absolutely does not have that trait.
50 = Completely neutral or average with regards to that trait.
100 = The face strongly displays that trait.

Clarification:
For hair-color, 0 = light, 100 = dark.
For gender, 0 = feminine, 100 = masculine.
"""

HUMAN_RATINGS_PATH = Path('/work/20260629-213612/omi-main/attribute_means.csv')

full_df = pd.read_csv(HUMAN_RATINGS_PATH)
pilot_df = full_df.sample(n=20, random_state=42)

claude = "anthropic/claude-sonnet-5"
gemini = "google/gemini-2.5-flash-lite"
gpt = "openai/gpt-5.4-mini"

collect_data(
    input_df=full_df,
    system_prompt=NEUTRAL_PROMPT,
    model_name=claude,
    prompt_label="neutral"
)

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=db20d035-0f07-44b3-9147-5cdb9d6d1fdd' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>